In [1]:
import pymrio
import pandas as pd
import os

CALC_PATH = 'D:/Programming/databases/IOT_2022_ixi_calc'
exio = pymrio.load_all(CALC_PATH)


In [13]:
import re
import numpy as np

# Build code → full EXIOBASE name mapping from classification file
clf = pd.read_excel(
    'D:/Programming/databases/IOT_2022_ixi/SI9 industry and product sector classification.xlsx',
    sheet_name='Industries', header=1
).dropna(subset=['CodeTxt'])
code_to_name = dict(zip(clf['CodeTxt'], clf['Name']))

# Reconcile against actual L index: classification uses "nec", L uses "n.e.c."
# Normalize by stripping dots to match across both conventions
L_sectors = exio.L.columns.get_level_values('sector').unique()
L_set = set(L_sectors)
L_norm = {s.replace('.', '').lower(): s for s in L_sectors}

for code, name in list(code_to_name.items()):
    if name not in L_set:
        norm_match = L_norm.get(name.replace('.', '').lower())
        if norm_match:
            code_to_name[code] = norm_match

# Load crosswalk, deduplicate on EXIOBASE Code
cw = pd.read_excel(
    'inputs/crosswalk_ISIC_Exiobase_260519.xlsx',
    sheet_name='AI attempt 2 Isic - exiobase'
)[['EXIOBASE Code', 'EXIOBASE Name']].drop_duplicates(subset='EXIOBASE Code')

sector_list = []
print(f"{'Code':<12} {'Short name':<30} Full EXIOBASE name")
print('-' * 95)
for _, row in cw.iterrows():
    code = row['EXIOBASE Code']
    full = code_to_name.get(code, None)
    if full is None:
        print(f"{code:<12} {str(row['EXIOBASE Name']):<30} SKIP — code not found in classification")
    else:
        sector_list.append(full)
        print(f"{code:<12} {str(row['EXIOBASE Name']):<30} {full}")

print(f"\n{len(sector_list)} sectors queued for processing.")

REGION = 'SE'

os.makedirs('outputs', exist_ok=True)
A_np = exio.A.fillna(0).to_numpy()
L_np = exio.L.fillna(0).to_numpy()
L_cols = exio.L.columns

Code         Short name                     Full EXIOBASE name
-----------------------------------------------------------------------------------------------
A_OTCR       Cultivation of crops nec       Cultivation of crops nec
A_FORE       Forestry                       Forestry, logging and related service activities (02)
A_FURN       Manufacture of furniture       Manufacture of furniture; manufacturing n.e.c. (36)
A_WOOD       Wood products                  Manufacture of wood and of products of wood and cork, except furniture; manufacture of articles of straw and plaiting materials (20)
A_TDWH       Wholesale trade                Wholesale trade and commission trade, except of motor vehicles and motorcycles (51)
A_TDRT       Retail trade                   Retail trade, except of motor vehicles and motorcycles; repair of personal and household goods (52)
A_HORE       Hotels & restaurants           Hotels and restaurants (55)
A_TAUX       Transport support              Supporting an

In [14]:
for sector_name in sector_list:
    key = (REGION, sector_name)
    if key not in L_cols:
        print(f'  SKIP: "{sector_name}" — not found in L index')
        continue

    y_np = np.zeros(len(L_cols))
    y_np[L_cols.get_loc(key)] = 1.0

    # full Leontief monetary requirements
    x_total = L_np @ y_np
    monetary_L = (pd.Series(x_total, index=exio.A.index)
                  .pipe(lambda s: s[s > 0])
                  .sort_values(ascending=False)
                  .reset_index())

    # tier decomposition
    monetary_tiers = {}
    x_tier = y_np.copy()
    for tier in range(10):
        monetary_tiers[tier] = x_tier
        x_tier = A_np @ x_tier

    monetary_df = pd.DataFrame(monetary_tiers, index=exio.A.index)
    monetary_df.columns.name = 'tier'
    monetary_total = monetary_df.sum(axis=1)
    monetary_out = (monetary_df
                    .loc[monetary_total[monetary_total > 0]
                         .sort_values(ascending=False).index]
                    .copy())
    monetary_out['total'] = monetary_out.sum(axis=1)
    monetary_out = monetary_out.reset_index()

    safe_name = re.sub(r'[^\w]+', '_', sector_name).strip('_')[:60]
    monetary_out.to_csv(f'outputs/{safe_name}_monetary_tier.csv', index=False)
    monetary_L.to_csv(f'outputs/{safe_name}_monetary_total.csv', index=False)
    print(f'{sector_name[:55]}: {len(monetary_out)} upstream rows')

Cultivation of crops nec: 6118 upstream rows
Forestry, logging and related service activities (02): 6118 upstream rows
Manufacture of furniture; manufacturing n.e.c. (36): 6118 upstream rows
Manufacture of wood and of products of wood and cork, e: 6118 upstream rows
Wholesale trade and commission trade, except of motor v: 6118 upstream rows
Retail trade, except of motor vehicles and motorcycles;: 6118 upstream rows
Hotels and restaurants (55): 6118 upstream rows
Supporting and auxiliary transport activities; activiti: 6118 upstream rows
Real estate activities (70): 6118 upstream rows
Other business activities (74): 6118 upstream rows
Publishing, printing and reproduction of recorded media: 6118 upstream rows
Computer and related activities (72): 6118 upstream rows
Financial intermediation, except insurance and pension : 6118 upstream rows
Education (80): 6118 upstream rows
Health and social work (85): 6118 upstream rows
Recreational, cultural and sporting activities (92): 6118 upstream